In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
import glob
import pandas as pd
from datetime import datetime
from sunpy.coordinates import sun

qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in ""


In [2]:
df = pd.read_csv('/home/ulyanov/data/solo/phi/wcs/fdt/wcs.csv', skipinitialspace=True).dropna()

dids = df['did'].to_numpy()
dates = np.array([datetime.fromisoformat(date) for date in df['date']])
sec = np.array([(datetime.fromisoformat(date) - dates[0]).total_seconds() for date in df['date']])

car_rot = df['car_rot'].to_numpy()
hgln = df['hgln_obs'].to_numpy()
crln = df['crln_obs'].to_numpy()
crlt = df['crlt_obs'].to_numpy()
dsun = df['dsun_au'].to_numpy()

b0 = np.array(sun.B0(dates))

x, y, z = np.cos(crlt * np.pi / 180) * np.cos(hgln * np.pi / 180), np.cos(crlt * np.pi / 180) * np.sin(hgln * np.pi / 180), np.sin(crlt * np.pi / 180)
xe, ye, ze = np.cos(b0 * np.pi / 180), 0, np.sin(b0 * np.pi / 180)
phi = np.arccos(x * xe + y * ye + z * ze) * 180 / np.pi

In [3]:
plt.rcParams['date.converter'] = 'concise'

t = np.where((car_rot[1:] - car_rot[:-1]) != 0)[0] + 1

fig, ax = plt.subplots(figsize=(10,8))
ax_ = ax.twinx()
ax1 = ax.twiny()


for a, b in zip(dates[t[::2]], dates[t[1::2]]):
    ax.axvspan(a, b, color='tab:blue', alpha=0.1)

ax.plot(dates, dsun, color='black', lw=1.25)
ax_.plot(dates, crlt, color='tab:red', lw=1.25)
#ax_.plot(dates, crln, color='tab:red', lw=1.25)

ax.set_xlim(datetime(2023,12,31), datetime(2026,1,1))
ax.set_ylim(0.2,1.)
ax_.set_ylim(-20,20)

ax_.tick_params(axis='y', colors='tab:red')

ax1.set_xlim(ax.get_xlim())
ax1.set_xticks(dates[t])
ax1.set_xticklabels(car_rot[t], rotation='vertical', size='small')

ax.set_xlabel('Date')
ax.set_ylabel('Distance, AU')
ax_.set_ylabel('Carrington latitude, degrees')
ax1.set_xlabel('Carrington rotation number')

ax.grid(True, axis='y')
plt.tight_layout()

In [4]:
plt.rcParams['date.converter'] = 'concise'

t = np.where((car_rot[1:] - car_rot[:-1]) != 0)[0] + 1

fig, ax = plt.subplots(figsize=(10,8))
ax_ = ax.twinx()
ax1 = ax.twiny()


for a, b in zip(dates[t[::2]], dates[t[1::2]]):
    ax.axvspan(a, b, color='tab:blue', alpha=0.1)

ax.plot(dates, hgln, color='black', lw=1.25)
ax.plot(dates, phi, '--', color='black', lw=1.25)
#ax_.plot(dates, crlt - b0, color='tab:red', lw=1.25)
ax_.plot(dates, b0, color='tab:red', lw=1.25)

ax.set_xlim(datetime(2023,12,31), datetime(2026,1,1))
ax.set_ylim(-200,200)
ax_.set_ylim(-40,40)

ax_.tick_params(axis='y', colors='tab:red')

ax1.set_xlim(ax.get_xlim())
ax1.set_xticks(dates[t])
ax1.set_xticklabels(car_rot[t], rotation='vertical', size='small')

ax.set_xlabel('Date')
ax.set_ylabel('Stonyhurst longitude, degrees')
ax_.set_ylabel(r'$\Delta B_0$, degrees')
ax1.set_xlabel('Carrington rotation number')
#plt.title('SOLO-Earth Separation angles')

ax.grid(True, axis='y')
plt.tight_layout()